In [15]:
import pandas as pd
import numpy as np
import itertools
import random

# ====================== LOAD CLUSTER DATA ======================
clusters = pd.read_csv('final_cluster_features.csv')   # ← your current cluster file

# ====================== USER PREFERENCE COMBINATIONS ======================
likes_options = [0, 1]
budget_options = [1, 2, 3]
days_options = list(range(1, 8))

user_combos = list(itertools.product(
    likes_options,   # Likes_Beach
    likes_options,   # Likes_Mountain
    likes_options,   # Likes_Culture
    likes_options,   # Likes_Adventure
    budget_options,
    days_options
))

print(f"Generating {len(user_combos)} user profiles × {len(clusters)} clusters = {len(user_combos)*len(clusters)} training rows")

# ====================== NEW SCORING FUNCTION (YOUR EXACT LOGIC) ======================
def compute_score(user, cluster):
    # 1. Preference Matching
    raw_match = 0.0
    selected_likes = 0
    missing_count = 0

    # Beach (ONLY Beach)
    if user['Likes_Beach'] == 1:
        selected_likes += 1
        beach_val = cluster.get('Beach', 0.0)
        raw_match += beach_val
        if beach_val == 0.0:
            missing_count += 1

    # Mountain
    if user['Likes_Mountain'] == 1:
        selected_likes += 1
        m_val = (cluster.get('Hiking', 0.0) +
                 cluster.get('Nature', 0.0) +
                 cluster.get('Viewpoint', 0.0))
        raw_match += m_val
        if m_val == 0.0:
            missing_count += 1
        if cluster.get('Park', 0.0) > 0:
            raw_match += 0.08   # fixed Park bonus

    # Culture
    if user['Likes_Culture'] == 1:
        selected_likes += 1
        c_val = (cluster.get('Culture', 0.0) +
                 cluster.get('History', 0.0) +
                 cluster.get('Religious', 0.0) +
                 cluster.get('Museum', 0.0) +
                 cluster.get('Architecture', 0.0))
        raw_match += c_val
        if c_val == 0.0:
            missing_count += 1

    # Adventure
    if user['Likes_Adventure'] == 1:
        selected_likes += 1
        a_val = (cluster.get('Adventure', 0.0) +
                 cluster.get('Safari', 0.0) +
                 cluster.get('Birding', 0.0) +
                 cluster.get('Hiking', 0.0))
        raw_match += a_val
        if a_val == 0.0:
            missing_count += 1

    # Shopping & Num_Places small bonuses
    if cluster.get('Shopping', 0.0) > 0:
        raw_match += 0.043
    raw_match += cluster.get('Num_Places', 0.0) * 0.043

    # Base matching score (max ~60)
    matching_score = min(60.0, raw_match * 70.0)

    # 2. Penalties
    penalty = -15.0 * missing_count
    if selected_likes == 1 and missing_count == 1:
        penalty -= 25.0   # extra strong penalty when user has only 1 like

    # 3. Completeness Bonus (+15 if ALL likes are covered at ≥10%)
    completeness_bonus = 0.0
    if selected_likes > 0 and missing_count == 0:
        all_covered = True
        
        if user['Likes_Beach'] == 1 and cluster.get('Beach', 0.0) < 0.10:
            all_covered = False
        if user['Likes_Mountain'] == 1 and (cluster.get('Hiking', 0.0) + cluster.get('Nature', 0.0) + cluster.get('Viewpoint', 0.0)) < 0.10:
            all_covered = False
        if user['Likes_Culture'] == 1 and (cluster.get('Culture', 0.0) + cluster.get('History', 0.0) + cluster.get('Religious', 0.0) + cluster.get('Museum', 0.0) + cluster.get('Architecture', 0.0)) < 0.10:
            all_covered = False
        if user['Likes_Adventure'] == 1 and (cluster.get('Adventure', 0.0) + cluster.get('Safari', 0.0) + cluster.get('Birding', 0.0) + cluster.get('Hiking', 0.0)) < 0.10:
            all_covered = False

        if all_covered:
            completeness_bonus = 15.0

    preference_total = matching_score + penalty + completeness_bonus

    # 4. Distance Score (max 25)
    closeness = 1 - cluster['Avg_Distance']
    days_factor = user['Total_Days'] / 7.0
    budget_factor = user['Budget'] / 3.0
    distance_score = closeness * 25 * days_factor * budget_factor

    # 5. Final Score
    score = preference_total + distance_score + random.uniform(-3, 3)
    return round(np.clip(score, 0, 100), 2)


# ====================== GENERATE TRAINING DATA ======================
training_rows = []

for combo in user_combos:
    user = {
        'Likes_Beach':    combo[0],
        'Likes_Mountain': combo[1],
        'Likes_Culture':  combo[2],
        'Likes_Adventure':combo[3],
        'Budget':         combo[4],
        'Total_Days':     combo[5]
    }
    
    for _, cluster_row in clusters.iterrows():
        cluster = cluster_row.to_dict()
        score = compute_score(user, cluster)
        
        row = {
            **user,
            **{col: cluster[col] for col in clusters.columns},
            'Score': score
        }
        training_rows.append(row)

training_df = pd.DataFrame(training_rows)
training_df = training_df.sample(frac=1, random_state=42).reset_index(drop=True)

# ====================== SAVE ======================
training_df.to_csv('training_dataset.csv', index=False)

print(f"✅ Training dataset successfully generated: {len(training_df)} rows")
print(f"File saved as: training_dataset.csv")
print("\nScore distribution:")
print(training_df['Score'].describe())
print("\nFirst 5 rows preview:")
print(training_df.head())

Generating 336 user profiles × 18 clusters = 6048 training rows
✅ Training dataset successfully generated: 6048 rows
File saved as: training_dataset.csv

Score distribution:
count    6048.000000
mean       47.391908
std        23.204588
min         0.000000
25%        33.010000
50%        49.540000
75%        64.285000
max        98.910000
Name: Score, dtype: float64

First 5 rows preview:
   Likes_Beach  Likes_Mountain  Likes_Culture  Likes_Adventure  Budget  \
0            1               0              0                1       2   
1            0               1              1                0       3   
2            1               1              0                0       1   
3            1               1              1                0       2   
4            1               0              0                1       3   

   Total_Days      Cluster_Name  Adventure  Architecture     Beach  ...  \
0           7     Dambulla Area        0.0      0.018519  0.000000  ...   
1           

In [16]:
training_df

,Likes_Beach,Likes_Mountain,Likes_Culture,Likes_Adventure,Budget,Total_Days,Cluster_Name,Adventure,Architecture,Beach,...,Nature,Park,Relax,Religious,Safari,Shopping,Viewpoint,Num_Places,Avg_Distance,Score
0,1,0,0,1,2,7,Dambulla Area,0.000000,0.018519,0.000000,...,0.166667,0.000000,0.055556,0.018519,0.111111,0.000000,0.055556,0.791667,0.427961,9.07
1,0,1,1,0,3,2,Kurunegala,0.000000,0.000000,0.000000,...,0.243243,0.027027,0.081081,0.216216,0.000000,0.027027,0.081081,0.437500,0.164374,81.36
2,1,1,0,0,1,2,Hambantota Area,0.000000,0.024390,0.048780,...,0.268293,0.000000,0.024390,0.146341,0.121951,0.000000,0.048780,0.520833,0.715727,27.46
3,1,1,1,0,2,3,Anuradhapura,0.000000,0.021739,0.000000,...,0.152174,0.021739,0.043478,0.173913,0.000000,0.021739,0.043478,0.625000,0.437584,46.51
4,1,0,0,1,3,5,Talaimannar Area,0.000000,0.000000,0.095238,...,0.142857,0.000000,0.047619,0.190476,0.095238,0.000000,0.095238,0.104167,0.614522,25.03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6043,1,0,0,1,3,7,Kandy Area,0.048780,0.048780,0.000000,...,0.121951,0.024390,0.024390,0.219512,0.000000,0.000000,0.073171,0.520833,0.246615,22.23
6044,1,1,0,1,3,2,Hatton Area,0.037037,0.000000,0.000000,...,0.333333,0.000000,0.129630,0.092593,0.000000,0.018519,0.074074,0.791667,0.263379,51.18
6045,1,1,0,1,3,4,Hambantota Area,0.000000,0.024390,0.048780,...,0.268293,0.000000,0.024390,0.146341,0.121951,0.000000,0.048780,0.520833,0.715727,47.28
6046,1,1,1,0,1,6,Jaffna Area,0.046512,0.000000,0.162791,...,0.186047,0.023256,0.046512,0.116279,0.000000,0.023256,0.093023,0.562500,1.000000,73.02
